<a href="https://colab.research.google.com/github/koushik-ace/NLP/blob/main/nlp_ass_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Contextual Word Embeddings and Text Classification using ELMo & BERT

### Part A: ELMo + Naive Bayes

#### A1: Load ELMo model

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import pandas as pd

In [ ]:
# Define the dataset
sentences = [
    "Win money now",
    "Claim your prize",
    "Hello how are you",
    "Let's meet tomorrow",
    "Free lottery ticket",
    "Are you coming today"
]
labels = [1, 1, 0, 0, 1, 0]

df = pd.DataFrame({'sentence': sentences, 'label': labels})
display(df.head())

,sentence,label
0,Win money now,1
1,Claim your prize,1
2,Hello how are you,0
3,Let's meet tomorrow,0
4,Free lottery ticket,1


Loading the ELMo model from TensorFlow Hub. This might take a moment to download.

In [ ]:
elmo_model = hub.load("https://tfhub.dev/google/elmo/3")
print("ELMo model loaded successfully!")

ELMo model loaded successfully!


#### A2: Generate word embeddings

In [ ]:
word_embeddings = elmo_model.signatures["default"](text=tf.constant(sentences))["elmo"]
print("Shape of word embeddings for each layer:")
for i, layer in enumerate(word_embeddings):
    print(f"Layer {i}: {layer.shape}")

Shape of word embeddings for each layer:
Layer 0: (4, 1024)
Layer 1: (4, 1024)
Layer 2: (4, 1024)
Layer 3: (4, 1024)
Layer 4: (4, 1024)
Layer 5: (4, 1024)


#### A3: Apply mean pooling

In [ ]:
# Retrieve the full output dictionary to get sequence lengths
elmo_output_dict = elmo_model.signatures["default"](
    text=tf.constant(sentences)
)

word_embeddings = elmo_output_dict["elmo"]
sequence_lengths = elmo_output_dict["sequence_len"]

# Create a mask for valid tokens based on actual sequence lengths
max_seq_len = tf.shape(word_embeddings)[1]
mask = tf.sequence_mask(sequence_lengths, maxlen=max_seq_len, dtype=tf.float32)
mask = tf.expand_dims(mask, -1) # Expand mask to match embedding dimension for broadcasting

# Apply mask: zero out embeddings of padding tokens
masked_embeddings = word_embeddings * mask

# Sum embeddings for each sentence along the sequence dimension
sum_embeddings = tf.reduce_sum(masked_embeddings, axis=1)

# Cast sequence_lengths to float32 for division and expand dimensions
actual_sentence_lengths = tf.cast(tf.expand_dims(sequence_lengths, -1), dtype=tf.float32)

# Compute mean embeddings by dividing sum by actual sentence length
sentence_embeddings = sum_embeddings / actual_sentence_lengths

print("Shape of sentence embeddings:", sentence_embeddings.shape)

Shape of sentence embeddings: (6, 1024)


#### A4: Train Naive Bayes model

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Split data into training and testing sets (though for a small dataset like this, it's mostly illustrative)
X_train, X_test, y_train, y_test = train_test_split(
    sentence_embeddings.numpy(), # Convert TensorFlow tensor to NumPy array
    df['label'].values, # Use labels from the DataFrame
    test_size=0.3, # 30% for testing
    random_state=42
)

# Initialize and train the Naive Bayes classifier
naive_bayes_classifier = GaussianNB()
naive_bayes_classifier.fit(X_train, y_train)

print("Naive Bayes classifier trained successfully!")

Naive Bayes classifier trained successfully!


#### A5: Evaluate performance

In [ ]:
# Make predictions on the test set
y_pred = naive_bayes_classifier.predict(X_test)

# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy on the test set: {accuracy:.4f}")

Accuracy on the test set: 0.0000


### Part B: BERT + Naive Bayes

#### B1: Load preprocessing and BERT model

In [ ]:
import tensorflow_hub as hub
import tensorflow_text as text # Required for BERT preprocessing ops

# Load the BERT preprocessor and encoder models
# You might need to change the model URLs based on your preference (e.g., small_bert/bert_en_uncased_L-2_H-128_A-2 for smaller models)
bert_preprocess_model = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder_model = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

print("BERT preprocessor and encoder models loaded successfully!")

#### B2: Preprocess input

In [ ]:
# Preprocess the sentences using the BERT preprocessor
bert_preprocessed_text = bert_preprocess_model(tf.constant(sentences))

print("BERT preprocessed input structure:")
for key, value in bert_preprocessed_text.items():
    print(f"  {key}: {value.shape}")

#### B3: Extract embeddings (pooled_output)

In [ ]:
# Pass the preprocessed input to the BERT encoder
bert_outputs = bert_encoder_model(bert_preprocessed_text)

# Extract the pooled_output, which is a fixed-size vector for the entire sequence
bert_sentence_embeddings = bert_outputs['pooled_output']

print("Shape of BERT sentence embeddings:", bert_sentence_embeddings.shape)

#### B4: Train Naive Bayes model

In [ ]:
# Split data into training and testing sets for BERT embeddings
X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    bert_sentence_embeddings.numpy(),
    df['label'].values,
    test_size=0.3,
    random_state=42
)

# Initialize and train a new Naive Bayes classifier for BERT embeddings
naive_bayes_classifier_bert = GaussianNB()
naive_bayes_classifier_bert.fit(X_train_bert, y_train_bert)

print("Naive Bayes classifier trained with BERT embeddings successfully!")

#### B5: Evaluate performance

In [ ]:
# Make predictions on the test set using the BERT-based model
y_pred_bert = naive_bayes_classifier_bert.predict(X_test_bert)

# Evaluate accuracy for BERT-based model
accuracy_bert = accuracy_score(y_test_bert, y_pred_bert)
print(f"Accuracy on the test set (BERT + Naive Bayes): {accuracy_bert:.4f}")